# Решения: частота и перебор

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

import matplotlib.pyplot as plt


def show_digit(row, title=''):
    """Одна строка таблицы -> картинка 8x8."""
    values = [int(v) for v in row[PIXELS]]
    grid = [values[i * 8:(i + 1) * 8] for i in range(8)]
    plt.imshow(grid, cmap='gray_r')
    plt.title(title)
    plt.axis('off')


## Урок. 1–2. Размер и картинка

In [ ]:
n_images, n_cols = df.shape
n_pixels = len(PIXELS)
n_classes = int(df['label'].nunique())
print(n_images, n_pixels, n_classes)
show_digit(df.loc[17], title=f"строка 17: цифра {df.loc[17, 'label']}")
n_zero_17 = int((df.loc[17, PIXELS] == 0).sum())
print('нулевых пикселей:', n_zero_17)

## Урок. 3–5. Частоты, baseline, событие «1 или 7»

In [ ]:
class_counts = df['label'].value_counts()
class_share = df['label'].value_counts(normalize=True)
top_digit = int(class_share.idxmax())
baseline_accuracy = float(class_share.max())
p_1_or_7 = float(df['label'].isin([1, 7]).mean())
print(class_share.sort_index().round(3))
print(top_digit, round(baseline_accuracy, 3), round(p_1_or_7, 3))

## Урок. 6–7. Перебор пар и настроек

In [ ]:
n_pairs = 0
for i in range(len(PIXELS)):
    for j in range(i + 1, len(PIXELS)):
        n_pairs += 1
n_pairs_formula = len(PIXELS) * (len(PIXELS) - 1) // 2
K_VALUES = [1, 3, 5, 7, 9]
FEATURE_SETS = ['все 64 пикселя', 'верхняя половина', 'сумма яркости']
configs = []
for k in K_VALUES:
    for fs in FEATURE_SETS:
        configs.append((k, fs))
print(n_pairs, n_pairs_formula, len(configs))

## Урок. 8–9. Смещённый поток и две картинки

In [ ]:
stream = df[df['label'].isin([0, 1])]
baseline_stream = float(stream['label'].value_counts(normalize=True).max())
BIAS_NOTE = (
    'На таком участке почти нет других цифр: глупый ответ «0» уже даёт ~50%. '
    'Точность, измеренная на равномерной таблице, не переносится на смещённый поток.'
)
p_same = float((df['label'].value_counts(normalize=True) ** 2).sum())
print(len(stream), round(baseline_stream, 3), round(p_same, 4))

## ДЗ. 1–4

In [ ]:
head = df.head(500)
share_500 = head['label'].value_counts(normalize=True)
top_500 = int(share_500.idxmax())
baseline_500 = float(share_500.max())
COMPARE_500 = (
    f'на 500 строках baseline {baseline_500:.3f}, '
    f"на всей таблице {df['label'].value_counts(normalize=True).max():.3f}"
)
n_codes = 0
for a in range(10):
    for b in range(10):
        for c in range(10):
            if a != b and b != c and a != c:
                n_codes += 1
WHY_BASELINE = (
    'Точность без опоры ни о чём не говорит: если 90% потока — цифра 1, '
    'распознаватель, который всегда отвечает 1, даёт 90% и не умеет ничего. '
    'Сравнение с baseline показывает, что модель добавила к простому правилу.'
)
print(top_500, round(baseline_500, 3), n_codes, len(WHY_BASELINE))